# VAA 4차 : aVAA 비교 + 후보 전체 튜닝 (메모리 절약판)

3차(`stuff_pipeline_261010_vaa3.ipynb`) 캐시를 그대로 쓰되, **필요한 열만** 읽고 점수 열은 float32 로 둔다. A·H1·H4·H5(1회 학습) 지표는 `comparison_round3.csv` 에서 읽는다.

aVAA (Lance Brozdowski) : 팔 슬롯·체격·익스텐션으로 예상되는 VAA 대비 얼마나 평평한가 (+ = 예상보다 평평). 회귀식 비공개
— [Lance's Pitcher Notes (2025-04-05)](https://lancebroz.substack.com/p/braves-spencer-schwellenbach-dbacks-brandon-pfaadt), [Bucs on Deck : VAAlHAAlla](https://bucsondeck.substack.com/p/vaalhaalla-a-deeper-dive-on-fastball)

구현 : `aVAA = vaa − E[vaa | 구종 ; plate_z, release_pos_z, release_extension, arm_angle, 투수 키]` (구종별 선형회귀 잔차, 2025 학습기간 적합). 높이도 함께 뺀다 (빼지 않으면 위치가 다시 들어옴).

**HAA 제외** (3차 : 어떤 방식이든 plate_x 누수 0.18 → 0.36~0.43, 성능 이득 없음). 비교 : H1v = F + vaa_adj, H5 = F + vaa_aa (Chamberlain), H6 = F + aVAA. 후보(H5·H6)는 F 와 같은 전체 튜닝(n_iter=12)으로 확인 (★)

In [ ]:
import os, gc, time, json, warnings, urllib.request, joblib
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count()))
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import stuff_pipeline as sp
import export_web_data as ew

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
SEED = sp.SEED
OUT = os.path.join('outputs', 'vaa_exp')
CACHE = os.path.join(OUT, 'cache')
T0 = time.time()
num = lambda s: s.to_numpy('float64', na_value=np.nan)

F_FEATS = [f for f in sp.STUFF_FEATURES if f not in ('vx0', 'vz0', 'ax', 'ay', 'az', 'vy0')]
WEB = ['pa_event', 'is_swing', 'is_whiff', 'result', 'xwoba', 'plate_z_norm']
COLS = sorted(set(F_FEATS) | {'pitcher', 'game_date', 'description', 'type', 'estimated_woba_using_speedangle', 'zone', 'plate_x', 'plate_z',
                              'plate_x_r', 'vaa', 'vaa_adj', 'waste', 'raw_F'} | set(WEB))
fr = {}
for y in (2024, 2025, 2026):
    cols = COLS + (['y3', 'split'] if y == 2025 else [])
    fr[y] = pd.read_parquet(os.path.join(CACHE, f'frame_{y}.parquet'), columns=cols)
    fr[y]['raw_F'] = fr[y]['raw_F'].astype('float32')
valid_start = pd.Timestamp(pd.read_json(os.path.join(CACHE, 'meta.json'), typ='series')['valid_start'])
is_tr = fr[2025]['game_date'] < valid_start
print('로드', {y: (len(f), f.shape[1]) for y, f in fr.items()}, f'메모리 약 {sum(f.memory_usage(deep=True).sum() for f in fr.values()) / 1e9:.2f} GB')

In [ ]:
# 투수 키 (3차에서 받아 둔 캐시, 없으면 MLB Stats API)
hpath = os.path.join(CACHE, 'heights.parquet')
if os.path.exists(hpath):
    heights = pd.read_parquet(hpath)['height_in']
else:
    def to_in(h):
        try:
            ft, inch = h.replace('"', '').split("'")
            return int(ft) * 12 + int(inch.strip() or 0)
        except Exception:
            return np.nan
    ids = sorted(set().union(*[set(fr[y]['pitcher'].astype(int)) for y in fr]))
    got = {}
    for i in range(0, len(ids), 150):
        with urllib.request.urlopen(f'{ew.STATSAPI}/people?personIds=' + ','.join(map(str, ids[i:i + 150])), timeout=30) as r:
            for pp in json.load(r).get('people', []):
                got[pp['id']] = to_in(pp.get('height', ''))
    heights = pd.Series(got, name='height_in')
    heights.to_frame().to_parquet(hpath)
ids = set().union(*[set(fr[y]['pitcher'].astype(int)) for y in fr])
print(f'투수 {len(ids):,}명 중 키 확인 {heights.reindex(list(ids)).notna().sum():,}명 · 평균 {heights.mean():.1f} in')
for y in fr:
    fr[y]['height_in'] = fr[y]['pitcher'].astype(int).map(heights).fillna(heights.mean()).astype('float64')

def fit_resid(df_fit, yname, xs):
    coef = {}
    for k, g in df_fit.groupby('pitch_type', observed=True):
        g = g[[yname] + xs].astype(float).dropna()
        if len(g) >= 500:
            A = np.column_stack([np.ones(len(g))] + [g[x].to_numpy() for x in xs])
            coef[str(k)] = np.linalg.lstsq(A, g[yname].to_numpy(), rcond=None)[0]
    return coef

def apply_resid(d, yname, xs, coef, out):
    res = np.full(len(d), np.nan)
    pt = d['pitch_type'].astype(str).to_numpy()
    X = np.column_stack([np.ones(len(d))] + [num(d[x]) for x in xs])
    yv = num(d[yname])
    for k, b in coef.items():
        m = pt == k
        res[m] = yv[m] - X[m] @ b
    d[out] = res

XS_AA = ['plate_z']
XS_AVAA = ['plate_z', 'release_pos_z', 'release_extension', 'arm_angle', 'height_in']
c_aa, c_av = fit_resid(fr[2025][is_tr], 'vaa', XS_AA), fit_resid(fr[2025][is_tr], 'vaa', XS_AVAA)
for y in fr:
    apply_resid(fr[y], 'vaa', XS_AA, c_aa, 'vaa_aa')
    apply_resid(fr[y], 'vaa', XS_AVAA, c_av, 'avaa')
d25 = fr[2025]
g = d25.groupby('pitch_type', observed=True)
display(pd.DataFrame({'n': g.size(), 'aVAA 평균': g['avaa'].mean(), 'aVAA 표준편차': g['avaa'].std(),
                      **{f'corr(aVAA, {x})': g.apply(lambda q, x=x: q['avaa'].corr(q[x].astype(float))) for x in XS_AVAA}}).query('n >= 1000').round(3))

In [ ]:
# 지표 함수 (3차와 같은 정의)
def reg_metrics(y, p):
    return {'RMSE': np.sqrt(mean_squared_error(y, p)), 'MAE': mean_absolute_error(y, p), 'R2': r2_score(y, p)}

def within_corr(d, col, loc, min_n=100):
    x = pd.DataFrame({'g1': d['pitcher'].to_numpy(), 'g2': d['pitch_type'].astype(str).to_numpy(), 'a': num(d[col]), 'b': num(d[loc])}).dropna()
    x = x.assign(ab=x['a'] * x['b'], aa=x['a'] ** 2, bb=x['b'] ** 2)
    s = x.groupby(['g1', 'g2']).agg(n=('a', 'size'), a=('a', 'sum'), b=('b', 'sum'), ab=('ab', 'sum'), aa=('aa', 'sum'), bb=('bb', 'sum'))
    s = s[s['n'] >= min_n]
    r = (s['ab'] - s['a'] * s['b'] / s['n']) / np.sqrt((s['aa'] - s['a'] ** 2 / s['n']) * (s['bb'] - s['b'] ** 2 / s['n']))
    ok = r.notna() & np.isfinite(r)
    return np.average(r[ok], weights=s['n'][ok])

def leak_r2(feats, target, n=200_000):
    tr = d25[is_tr & d25[target].notna()].sample(n, random_state=SEED)
    va = d25[~is_tr & d25[target].notna()]
    m = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.08, num_leaves=63, random_state=SEED, verbose=-1).fit(tr[feats], tr[target])
    return r2_score(va[target], m.predict(va[feats]))

def validity(d, col):
    d = d.assign(swing=d['description'].isin(ew.SWING_DESC), whiff=d['description'].isin(sp.WHIFF_DESC), xc=d['estimated_woba_using_speedangle'].where(d['type'] == 'X'))
    pp = d.groupby(['pitcher', 'pitch_type'], observed=True).agg(n=('swing', 'size'), s=(col, 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum'), xc=('xc', 'mean')).query('n >= 100')
    pt = d.groupby('pitcher').agg(n=('swing', 'size'), s=(col, 'mean'), sw=('swing', 'sum'), wh=('whiff', 'sum')).query('n >= 300')
    return {'투수×구종 ρ(구위, whiff%)': stats.spearmanr(pp['s'], pp['wh'] / pp['sw']).statistic,
            '투수×구종 ρ(구위, xwOBAcon)': stats.spearmanr(pp['s'], pp['xc'], nan_policy='omit').statistic,
            '투수 ρ(구위, whiff%)': stats.spearmanr(pt['s'], pt['wh'] / pt['sw']).statistic}

def pt_means(d, col, min_n=100):
    g = d.assign(pitch_type=d['pitch_type'].astype(str)).groupby(['pitcher', 'pitch_type'])[col].agg(['mean', 'count'])
    return g[g['count'] >= min_n]['mean']

def pt_whiff(d, min_n=100):
    g = d.assign(pitch_type=d['pitch_type'].astype(str), sw=d['description'].isin(ew.SWING_DESC), wh=d['description'].isin(sp.WHIFF_DESC)) \
         .groupby(['pitcher', 'pitch_type']).agg(n=('sw', 'size'), sw=('sw', 'sum'), wh=('wh', 'sum'))
    g = g[g['n'] >= min_n]
    return g['wh'] / g['sw']

def buckets(d, col):
    bins = pd.cut(d[col], ew.STUFF_BINS, labels=ew.STUFF_BIN_LABELS, right=False)
    return pd.DataFrame({lab: ew.outcome_rates(d[bins == lab]) for lab in ew.STUFF_BIN_LABELS}).T

samp = d25[d25['y3'].notna()]
fit_rows, va_rows = samp[samp['split'] != 'va'], samp[samp['split'] == 'va']
tr_rows, es_rows = samp[samp['split'] == 'tr'], samp[samp['split'] == 'es']
d26 = fr[2026]
aux26 = pd.DataFrame({'abs_x': d26['plate_x'].abs(), 'ball': d26['description'].isin(sp.BALL_DESC),
                      'hbp': d26['description'] == 'hit_by_pitch', 'out_zone': d26['zone'] >= 11}, index=d26.index)
PTS = ['FF', 'SI', 'FC', 'SL', 'ST', 'CH', 'FS', 'CU', 'KC']
model_F = joblib.load(os.path.join(OUT, 'stage3_F.joblib'))
FEATS = {'F': F_FEATS, 'H1v': F_FEATS + ['vaa_adj'], 'H5': F_FEATS + ['vaa_aa'], 'H6': F_FEATS + ['avaa']}
results, bias = {}, {}

def evaluate(k, m, feats):
    # 점수 열을 만들고 지표 계산 후, 필요 없는 열은 지운다 (메모리)
    sc = sp.StuffScaler().fit(m.predict(d25.loc[is_tr & ~d25['waste'], feats])) if k != 'F' else joblib.load(os.path.join(OUT, 'stage4_F.joblib'))
    for y in fr:
        raw = sc.transform(m.predict(fr[y][feats])).astype('float32')
        fr[y]['_raw'] = raw
        fr[y]['_s'] = pd.Series(raw, index=fr[y].index).where(~fr[y]['waste'])
    r = {}
    for a, v in reg_metrics(va_rows['y3'], m.predict(va_rows[feats])).items():
        r[f'검증 {a} (공통 표본)'] = v
    for a, v in validity(d25[~is_tr], '_s').items():
        r[f'검증 {a}'] = v
    for year in (2024, 2026):
        b = buckets(fr[year], '_s')
        r[f'{year} BA 격차'] = b.loc['<40', 'ba'] - b.loc['60+', 'ba']
        r[f'{year} Whiff 격차'] = b.loc['60+', 'whiff'] - b.loc['<40', 'whiff']
    for a, b in [(2024, 2025), (2025, 2026)]:
        mm = pd.concat([pt_means(fr[a], '_s'), pt_means(fr[b], '_s')], axis=1, join='inner')
        r[f'안정성 r(구위 {a}, 구위 {b})'] = mm.corr().iloc[0, 1]
        w = pd.concat([pt_means(fr[a], '_s'), pt_whiff(fr[b])], axis=1, join='inner')
        r[f'예측력 ρ(구위 {a}, whiff% {b})'] = stats.spearmanr(w.iloc[:, 0], w.iloc[:, 1]).statistic
    r['누수 R²(plate_z)'] = leak_r2(feats, 'plate_z')
    r['누수 R²(plate_x 우투 기준)'] = leak_r2(feats, 'plate_x_r')
    e26 = d26.assign(abs_x=aux26['abs_x'])
    r['투수×구종 내 corr(점수, 높이 정규화)'] = within_corr(e26, '_s', 'plate_z_norm')
    r['투수×구종 내 corr(점수, |plate_x|)'] = within_corr(e26, '_s', 'abs_x')
    top = d26['_raw'] >= d26['_raw'].quantile(0.99)
    r['사구 평균 점수 (raw)'] = d26.loc[aux26['hbp'], '_raw'].mean()
    r['상위 1% 중 존 밖 볼 비율 (raw)'] = (aux26.loc[top, 'ball'] & aux26.loc[top, 'out_zone']).mean()
    bias[k] = {pt: within_corr(d26[d26['pitch_type'] == pt], '_s', 'plate_z_norm') for pt in PTS}
    c = m.booster_.predict(va_rows[feats].sample(5000, random_state=SEED), pred_contrib=True)[:, :-1]
    names = [{'vaa_aa': 'vaa', 'avaa': 'vaa', 'vaa_adj': 'vaa'}.get(f, f) for f in feats]
    for gname, cols in ew.EXPLAIN_GROUPS:
        idx = [i for i, f in enumerate(names) if f in cols]
        r[f'|SHAP|×1000 {gname}'] = np.abs(c[:, idx].sum(axis=1)).mean() * 1000 if idx else np.nan
    for y in fr:
        fr[y].drop(columns=['_raw', '_s'], inplace=True)
    gc.collect()
    results[k] = pd.Series(r)

In [ ]:
t = time.time()
params = model_F.get_params()
evaluate('F', model_F, F_FEATS)                      # 재현 검사용
for k in ['H1v', 'H6']:
    m = lgb.LGBMRegressor(**params).fit(fit_rows[FEATS[k]], fit_rows['y3'])
    joblib.dump(m, os.path.join(OUT, f'stage3_{k}.joblib'))
    evaluate(k, m, FEATS[k])
print(f'1회 학습 변형 {time.time() - t:.0f}초')

### 후보 전체 튜닝 (F 와 같은 절차 : sp.fit_stuff_model, n_iter = 12) — 하나씩 순서대로

In [ ]:
for k in ['H5', 'H6']:
    t = time.time()
    m = sp.fit_stuff_model(tr_rows[FEATS[k]], tr_rows['y3'], es_rows[FEATS[k]], es_rows['y3'], n_iter=12, verbose=False)
    joblib.dump(m, os.path.join(OUT, f'stage3_{k}t.joblib'))
    print(f'{k}★ best holdout rmse {m.tuning_results_["rmse"].min():.5f} (F 0.16879) · {time.time() - t:.0f}초')
    evaluate(f'{k}★', m, FEATS[k])
    del m
    gc.collect()

In [ ]:
r3 = pd.read_csv(os.path.join(OUT, 'comparison_round3.csv'), index_col=0, encoding='utf-8-sig')
ref = r3[['A v2 (웹)', 'H1 F+회귀 보정', 'H4 F+Chamberlain VAA·HAA', 'H5 F+Chamberlain VAA만', 'H3 F+기준점 VAA만']]
new = pd.DataFrame(results).rename(columns={'F': 'F (재계산)', 'H1v': 'H1v F+회귀 보정 VAA', 'H6': 'H6 F+aVAA',
                                             'H5★': 'H5★ Chamberlain VAA (튜닝)', 'H6★': 'H6★ aVAA (튜닝)'})
cmp4 = pd.concat([ref, new], axis=1).reindex(new.index)
print('재현 검사 : F 재계산 − 3차 F (최대 절대 차이)', float((new['F (재계산)'] - r3['F']).reindex(new.index).dropna().abs().max()))
display(cmp4.round(4))
cmp4.to_csv(os.path.join(OUT, 'comparison_round4.csv'), encoding='utf-8-sig')
b = pd.DataFrame(bias)
display(b.round(3))
b.to_csv(os.path.join(OUT, 'height_bias_round4.csv'), encoding='utf-8-sig')
print(f'전체 {time.time() - T0:.0f}초')

### 판정 (2026-10-10)
> 이 노트북은 nbconvert 커널 연결 문제(로컬 통신 차단 추정)로 같은 코드를 일반 파이썬 스크립트로 실행했다 (198초). 출력은 `outputs/vaa_exp/comparison_round4.csv`, `height_bias_round4.csv` 에 있다. F 재계산 값이 3차와 같음 (최대 차이 9.5e-7)

- aVAA : 구종 안에서 plate_z·릴리스 높이·익스텐션·팔각도·키와 상관 ≈ 0, 표준편차 FF 0.27° ~ CU 0.77°. 투수 키 1,299명 모두 확인
- F 계열(F, H1v, H5, H6, 튜닝판 H5★·H6★)은 모든 지표가 오차 범위 안에서 같다 : ρ(whiff%) 0.717~0.729, 안정성 0.93~0.94, 다음 시즌 whiff% ρ 0.77~0.78, 상위 1% 존 밖 볼 41~42%
- 차이 : 높이 누수 R² F 0.28 < H5 0.30 < H6 0.35. 좌우 누수는 VAA 만 넣은 버전 모두 0.18 (HAA 를 넣은 H1·H4 는 0.36~0.38)
- 전체 튜닝(★)으로도 결론이 바뀌지 않음

**최선 : H5★ (F + Chamberlain 방식 VAA Above Average)**
1. 목적 적합 : 위치 의존성이 F 와 같은 수준 (A 대비 좌우 의존 0.43 → 0.02, 고득점 존 밖 볼 93% → 42%)
2. 실력 측정 : 안정성·다음 시즌 예측이 A 보다 크게 좋고 F 계열 최상위권
3. 해석 : '진입각' 요인으로 근거 표시 가능, FanGraphs 에 공개된 개념(VAA Above Average)이라 설명 가능
4. aVAA(H6) 대비 : 성능 같고 높이 누수 낮음. aVAA 의 '팔 높이·익스텐션 예상치'는 모델에 이미 입력으로 있어 중복이고, 투수 키 외부 의존(MLB API)이 생김
5. 대가(A 대비) : 투구 단위 적합도·구간 격차 감소 = 위치 덕분의 맞힘이 빠진 것
운영 반영은 사용자 지시 후